# Audio Demo: Listen to Moshi Self-Play

**Goal:** Load any Moshi variant (teacher, pruned, post-KD), run self-play for
a configurable duration, listen to the output inline, and save to GDrive.

**Workflow:**
1. Select a model variant from the dropdown
2. Configure duration, seed, and generation params
3. Run self-play generation
4. Listen to the generated self-play audio
5. (Optional) Compare side-by-side with another variant
6. Save audio + metadata to GDrive

**Run on:** Colab Free (T4) for pruned variants, Colab Pro (L4/A100) for full teacher

**Note on self-play audio:** In self-play mode, the model talks to itself —
its output at step *t* is fed back as input at step *t+1*. There is only one
speaker (the model). The audio you hear IS the model's speech quality.

## Cell 1: Session Startup

In [ ]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

# Construct URL with auth token
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = "/content/moshilite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## Cell 2: Configuration

Use the Colab form widgets (right panel) to select model variant, duration, seed, etc.

**Model categories:**
- `teacher` = Full Moshi baseline (no pruning)
- `*_30pct` / `v*_nonuni` / `v*_uni` = Pruned only (pre-KD, NOT trained)
- `*_L{1-5}_kd` = Pruned AND KD-trained
- `custom` = Any arbitrary .pt checkpoint path

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  MODEL SELECTION                                                     ║
# ║                                                                       ║
# ║  Categories:                                                          ║
# ║    teacher              = Full Moshi baseline (no pruning)            ║
# ║    *_30pct / v*_nonuni  = Pruned only (pre-KD, NOT trained)           ║
# ║    *_L{1-5}_kd          = Pruned AND KD-trained                       ║
# ║    custom               = Any arbitrary .pt checkpoint path           ║
# ╚══════════════════════════════════════════════════════════════════════╝

MODEL_VARIANT = "teacher"  # @param ["teacher", "magnitude_30pct", "wanda_30pct", "sparsegpt_30pct", "v1_scattered_nonuni", "v1_scattered_uni", "v2a_cont_strict_nonuni", "v2b_cont_penalized_nonuni", "v2c_cont_relaxed_nonuni", "v3_collapse_nonuni", "magnitude_30pct_L1_kd", "magnitude_30pct_L2_kd", "wanda_30pct_L1_kd", "wanda_30pct_L2_kd", "sparsegpt_30pct_L1_kd", "sparsegpt_30pct_L2_kd", "v1_scattered_nonuni_L1_kd", "v1_scattered_nonuni_L2_kd", "v1_scattered_nonuni_L3_kd", "v1_scattered_nonuni_L4_kd", "v1_scattered_nonuni_L5_kd", "custom"]

# If MODEL_VARIANT == "custom", specify the full path to a .pt checkpoint:
CUSTOM_CHECKPOINT_PATH = ""  # @param {type:"string"}

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  GENERATION SETTINGS                                                ║
# ╚══════════════════════════════════════════════════════════════════════╝

DURATION_SECONDS = 30  # @param {type:"slider", min:5, max:300, step:5}
SEED = 42              # @param {type:"integer"}
SEED_TYPE = "noise"    # @param ["noise", "acoustic", "silence"]
TEMPERATURE = 0.8      # @param {type:"number"}
TEMPERATURE_TEXT = 0.7 # @param {type:"number"}
REPETITION_PENALTY = 1.3  # @param {type:"number"}

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PATHS (auto-derived — edit EXPERIMENT_ID if needed)                ║
# ╚══════════════════════════════════════════════════════════════════════╝

EXPERIMENT_ID = "prune30_v1"
GDRIVE_BASE = "/content/drive/MyDrive/moshilite"
WEIGHTS_DIR = f"{GDRIVE_BASE}/upstream_weights/moshiko"
CHECKPOINT_DIR = f"{GDRIVE_BASE}/checkpoints/{EXPERIMENT_ID}"
MODELS_DIR = f"{GDRIVE_BASE}/models"
AUDIO_SAMPLES_DIR = f"{GDRIVE_BASE}/audio_samples"

# ── Resolve model path ──
from pathlib import Path

def resolve_model_path(variant):
    """Map variant name to checkpoint path."""
    if variant == "teacher":
        return WEIGHTS_DIR, "teacher"
    elif variant == "custom":
        if not CUSTOM_CHECKPOINT_PATH:
            raise ValueError("Set CUSTOM_CHECKPOINT_PATH when using 'custom' variant")
        return CUSTOM_CHECKPOINT_PATH, "custom"
    elif variant.endswith("_kd"):
        exported = Path(f"{MODELS_DIR}/{variant}.pt")
        if exported.exists():
            return str(exported), "post_kd"
        run_id = variant.removesuffix("_kd")
        best_ckpt = Path(f"{CHECKPOINT_DIR}/kd/{run_id}/checkpoint_best.pt")
        if best_ckpt.exists():
            return str(best_ckpt), "post_kd"
        return str(exported), "post_kd"
    else:
        path = f"{CHECKPOINT_DIR}/{variant}.pt"
        return path, "pruned"

MODEL_PATH, MODEL_TYPE = resolve_model_path(MODEL_VARIANT)
NUM_STEPS = int(DURATION_SECONDS * 12.5)

# ── Auto-discover available models on GDrive ──
print("Available models on GDrive:")
print()

teacher_dir = Path(WEIGHTS_DIR)
if teacher_dir.exists():
    print("  [Teacher]")
    print(f"    teacher")

ckpt_dir = Path(CHECKPOINT_DIR)
if ckpt_dir.exists():
    pruned_files = sorted(ckpt_dir.glob("*.pt"))
    if pruned_files:
        print("  [Pruned only — pre-KD, NOT trained]")
        for f in pruned_files:
            size_gb = f.stat().st_size / 1e9
            print(f"    {f.stem:40s} ({size_gb:.1f} GB)")

models_dir = Path(MODELS_DIR)
kd_found = False
if models_dir.exists():
    kd_files = sorted(models_dir.glob("*_kd.pt"))
    if kd_files:
        kd_found = True
        print("  [Pruned + KD trained — exported]")
        for f in kd_files:
            size_gb = f.stat().st_size / 1e9
            print(f"    {f.stem:40s} ({size_gb:.1f} GB)")

kd_train_dir = ckpt_dir / "kd"
if kd_train_dir.exists():
    kd_runs = sorted(kd_train_dir.iterdir())
    unexported = []
    for run_dir in kd_runs:
        if run_dir.is_dir() and (run_dir / "checkpoint_best.pt").exists():
            run_id = run_dir.name
            export_name = f"{run_id}_kd"
            if not (models_dir / f"{export_name}.pt").exists():
                unexported.append(export_name)
    if unexported:
        kd_found = True
        print("  [Pruned + KD trained — NOT exported, using checkpoint_best.pt]")
        for name in unexported:
            print(f"    {name}")

if not kd_found:
    print("  [Pruned + KD trained]: none found")

print()
print(f"{'='*62}")
print(f"  Selected Configuration")
print(f"{'='*62}")
print(f"  Model:      {MODEL_VARIANT}")
print(f"  Type:       {MODEL_TYPE}")
print(f"  Path:       {MODEL_PATH}")
print(f"  Duration:   {DURATION_SECONDS}s ({NUM_STEPS} steps)")
print(f"  Seed:       {SEED} ({SEED_TYPE})")
print(f"  Temp:       {TEMPERATURE} (audio) / {TEMPERATURE_TEXT} (text)")
print(f"  Rep. pen.:  {REPETITION_PENALTY}")
print(f"{'='*62}")

if MODEL_VARIANT != "teacher" and not Path(MODEL_PATH).exists():
    print(f"\n  WARNING: {MODEL_PATH} not found!")
    print(f"    Available options are listed above. Update MODEL_VARIANT and re-run.")

## Cell 3: Load Model + Mimi Codec

Loads the selected LM model **and** the Mimi audio codec (for decoding tokens to waveform).

In [ ]:
import torch
import gc
import json
from pathlib import Path
from moshi.models import loaders
from moshi.models.lm import LMGen

# ── Detect dtype ──
gpu_name = torch.cuda.get_device_name(0).lower()
if any(x in gpu_name for x in ["a100", "h100", "l4", "l40"]):
    DTYPE = torch.bfloat16
    print(f"Using bfloat16 (detected {gpu_name})")
else:
    DTYPE = torch.float16
    print(f"Using float16 (detected {gpu_name})")

# ── Load LM model ──
print(f"\n{'='*60}")
print(f"  Loading LM: {MODEL_VARIANT}")
print(f"{'='*60}")

weights_path = Path(MODEL_PATH)

if MODEL_VARIANT == "teacher":
    config_path = weights_path / "config.json"
    moshi_name = "model.safetensors"
    config = None

    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)
        moshi_name = config.get("moshi_name", moshi_name)
    else:
        print("  No config.json — using default Moshiko architecture")

    model_path = weights_path / moshi_name
    if not model_path.exists():
        sf_files = list(weights_path.glob("*.safetensors"))
        sf_files = [f for f in sf_files if "tokenizer" not in f.name.lower() and "mimi" not in f.name.lower()]
        if sf_files:
            model_path = sf_files[0]
        else:
            raise FileNotFoundError(f"No LM weights found in {weights_path}")
    print(f"  Weights file: {model_path.name}")

    lm_model = loaders.get_moshi_lm(
        model_path, lm_kwargs=config, device="cuda", dtype=DTYPE
    )
    variant_meta = {"pruning_type": "none", "sparsity": 0.0, "kd_trained": False}

else:
    if not weights_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {weights_path}")

    print(f"  Loading checkpoint: {weights_path.name}")
    ckpt = torch.load(str(weights_path), map_location="cpu", weights_only=False)

    variant_meta = {
        "pruning_type": ckpt.get("pruning_type", "structured"),
        "pruning_method": ckpt.get("pruning_method", "unknown"),
        "sparsity": ckpt.get("actual_sparsity", ckpt.get("sparsity", 0.0)),
        "kd_trained": MODEL_TYPE == "post_kd",
        "loss_config": ckpt.get("config", {}).get("loss_config", None) if isinstance(ckpt.get("config"), dict) else None,
        "param_billions": ckpt.get("param_billions", None),
        "nonzero_param_billions": ckpt.get("nonzero_param_billions", None),
    }

    upstream_path = Path(WEIGHTS_DIR)
    up_config_path = upstream_path / "config.json"
    up_config = None
    up_moshi_name = "model.safetensors"

    if up_config_path.exists():
        with open(up_config_path) as f:
            up_config = json.load(f)
        up_moshi_name = up_config.get("moshi_name", up_moshi_name)

    up_model_path = upstream_path / up_moshi_name
    if not up_model_path.exists():
        sf_files = list(upstream_path.glob("*.safetensors"))
        sf_files = [f for f in sf_files if "tokenizer" not in f.name.lower() and "mimi" not in f.name.lower()]
        if sf_files:
            up_model_path = sf_files[0]

    lm_model = loaders.get_moshi_lm(
        up_model_path, lm_kwargs=up_config, device="cuda", dtype=DTYPE
    )

    state_dict = ckpt.get("model_state_dict", ckpt)
    missing, unexpected = lm_model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  Missing keys: {len(missing)} (may be expected for structured pruning)")
    if unexpected:
        print(f"  Unexpected keys: {len(unexpected)}")
    print(f"  Pruned weights overlaid successfully")
    del ckpt, state_dict

lm_model.eval()
n_params = sum(p.numel() for p in lm_model.parameters()) / 1e9
n_nonzero = sum((p != 0).sum().item() for p in lm_model.parameters()) / 1e9
print(f"\n  Model info:")
print(f"    Total params:    {n_params:.3f}B")
print(f"    Non-zero params: {n_nonzero:.3f}B")
print(f"    n_q={lm_model.n_q}, dep_q={lm_model.dep_q}, dim={lm_model.dim}")
print(f"    VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Load Mimi codec ──
print(f"\n{'='*60}")
print(f"  Loading Mimi Audio Codec")
print(f"{'='*60}")

mimi_candidates = (
    list(Path(WEIGHTS_DIR).glob("tokenizer*.safetensors"))
    + list(Path(WEIGHTS_DIR).glob("mimi*.safetensors"))
)
if mimi_candidates:
    mimi_path = str(mimi_candidates[0])
    print(f"  Using: {Path(mimi_path).name}")
else:
    mimi_path = None
    print("  No local Mimi weights — will download from HF")

mimi = loaders.get_mimi(mimi_path, device="cuda")
mimi.eval()
SAMPLE_RATE = mimi.sample_rate
print(f"  Mimi loaded (sample_rate={SAMPLE_RATE})")
print(f"  Total VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Create LMGen ──
lm_gen = LMGen(
    lm_model,
    use_sampling=True,
    temp=TEMPERATURE,
    temp_text=TEMPERATURE_TEXT,
    top_k=250,
    top_k_text=25,
)
print(f"\nReady to generate! ({MODEL_VARIANT})")

## Cell 4: Run Self-Play Generation

Runs the model in self-play mode for the configured duration.
The model's output at each step is fed back as input for the next step.

In [ ]:
import time
import numpy as np

print(f"Generating {DURATION_SECONDS}s self-play...")
print(f"   Model: {MODEL_VARIANT} | Seed: {SEED} ({SEED_TYPE}) | Steps: {NUM_STEPS}")
print()

dep_q = lm_model.dep_q
n_user_cb = lm_model.n_q - dep_q
card = lm_model.card

# ── Generate seed tokens ──
rng = torch.Generator(device="cpu").manual_seed(SEED)
if SEED_TYPE == "noise":
    seed_tokens = torch.randint(0, card, (1, n_user_cb, 1), generator=rng).cuda()
elif SEED_TYPE == "acoustic":
    seed_tokens = torch.randint(0, card // 4, (1, n_user_cb, 1), generator=rng).cuda()
elif SEED_TYPE == "silence":
    seed_tokens = torch.zeros(1, n_user_cb, 1, dtype=torch.long).cuda()
else:
    raise ValueError(f"Unknown seed type: {SEED_TYPE}")

# ── Repetition penalty ──
recent_text_tokens = []

def text_rep_penalty_hook(logits):
    if not recent_text_tokens or REPETITION_PENALTY <= 1.0:
        return
    logit_vec = logits[0, 0, 0]
    for tok in set(recent_text_tokens):
        if tok < logit_vec.shape[0]:
            if logit_vec[tok] > 0:
                logit_vec[tok] /= REPETITION_PENALTY
            else:
                logit_vec[tok] *= REPETITION_PENALTY

if REPETITION_PENALTY > 1.0:
    lm_gen.on_text_logits_hook = text_rep_penalty_hook

# ── Self-play loop ──
all_moshi_codes = []
all_text_tokens = []

t_start = time.time()

try:
    with torch.no_grad(), lm_gen.streaming(batch_size=1):
        prev_model_audio = seed_tokens

        for t in range(NUM_STEPS):
            out = lm_gen.step(prev_model_audio)

            if out is None:
                continue

            out_squeezed = out[0, :, 0]
            text_tok = out_squeezed[0].item()
            model_audio = out_squeezed[1:]

            if text_tok > 3:
                recent_text_tokens.append(text_tok)
                if len(recent_text_tokens) > 50:
                    recent_text_tokens.pop(0)

            all_moshi_codes.append(model_audio.unsqueeze(-1))
            all_text_tokens.append(text_tok)

            # Self-play: feed output back as input
            prev_model_audio = model_audio.unsqueeze(0).unsqueeze(-1).long()

            if (t + 1) % 62 == 0:
                elapsed = time.time() - t_start
                audio_s = len(all_moshi_codes) / 12.5
                print(f"   [{audio_s:.0f}s / {DURATION_SECONDS}s] "
                      f"{elapsed:.1f}s elapsed, "
                      f"{elapsed / max(1, t+1) * 1000:.0f} ms/step")

finally:
    lm_gen.on_text_logits_hook = None

t_elapsed = time.time() - t_start

if not all_moshi_codes:
    raise RuntimeError("No valid output produced. Try more steps.")

moshi_codes = torch.cat(all_moshi_codes, dim=-1).unsqueeze(0)  # [1, 8, T]
text_tokens_arr = np.array(all_text_tokens, dtype=np.int32)

actual_duration = moshi_codes.shape[-1] / 12.5
ms_per_step = (t_elapsed / NUM_STEPS) * 1000
rtf = actual_duration / t_elapsed if t_elapsed > 0 else 0

print(f"\nGeneration complete!")
print(f"   Audio duration: {actual_duration:.1f}s")
print(f"   Wall time:      {t_elapsed:.1f}s")
print(f"   Speed:          {ms_per_step:.0f} ms/step")
print(f"   Real-time:      {rtf:.2f}x")
print(f"   Codes shape:    {list(moshi_codes.shape)}")

## Cell 5: Decode & Play Audio

Decodes audio tokens to a waveform using the Mimi codec, then plays inline.

**VRAM note:** The LM model is moved to CPU before decoding to free GPU memory.
Re-run Cell 3 if you want to regenerate with different settings.

In [ ]:
import gc
import IPython.display as ipd
from IPython.display import HTML, display

# ── Free VRAM: move LM to CPU before decoding ──
moshi_codes_cpu = moshi_codes.cpu()

print("Moving LM model to CPU to free VRAM for Mimi decode...")
lm_model.cpu()
del lm_gen
gc.collect()
torch.cuda.empty_cache()
print(f"  VRAM after offload: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ── Decode in chunks to avoid OOM ──
CHUNK_TOKENS = 125  # ~10 seconds per chunk

def decode_chunked(codec, codes_cpu, chunk_size=CHUNK_TOKENS):
    """Decode audio codes in chunks to avoid OOM."""
    total_t = codes_cpu.shape[-1]
    wav_chunks = []
    for start in range(0, total_t, chunk_size):
        end = min(start + chunk_size, total_t)
        chunk = codes_cpu[:, :, start:end].cuda()
        with torch.no_grad():
            wav_chunk = codec.decode(chunk)
        wav_chunks.append(wav_chunk.cpu())
        del chunk
        torch.cuda.empty_cache()
    return torch.cat(wav_chunks, dim=-1)

print("Decoding audio tokens with Mimi...")
moshi_wav = decode_chunked(mimi, moshi_codes_cpu)
print(f"  Decoded: {moshi_wav.shape}")

# Extract numpy
moshi_audio = moshi_wav[0, 0].numpy()

# Normalize
def normalize_audio(audio):
    peak = max(abs(audio.max()), abs(audio.min()), 1e-8)
    if peak > 1.0:
        return audio / peak * 0.95
    return audio

moshi_audio = normalize_audio(moshi_audio)

# ── Decode text tokens ──
unique_text = [t for t in text_tokens_arr if t > 3]
n_unique = len(set(unique_text))
n_meaningful = len(unique_text)
pad_ratio = sum(1 for t in text_tokens_arr if t <= 3) / max(1, len(text_tokens_arr))

text_transcript = ""
try:
    from sentencepiece import SentencePieceProcessor
    tokenizer_path = Path(WEIGHTS_DIR) / "tokenizer_spm_32k_3.model"
    if tokenizer_path.exists():
        sp = SentencePieceProcessor(str(tokenizer_path))
        meaningful_tokens = [int(t) for t in text_tokens_arr if t > 3]
        if meaningful_tokens:
            text_transcript = sp.decode(meaningful_tokens)
except Exception as e:
    text_transcript = f"(tokenizer unavailable: {e})"

# ── Display info ──
display(HTML(f"""
<div style="background: #1a1a2e; color: #eee; padding: 20px; border-radius: 12px; margin: 10px 0; font-family: 'Segoe UI', sans-serif;">
  <h2 style="color: #e94560; margin-top: 0;">Self-Play Audio: {MODEL_VARIANT}</h2>
  <table style="width: 100%; color: #ccc; font-size: 14px;">
    <tr><td style="padding: 2px 10px;">Duration</td><td>{actual_duration:.1f}s ({len(moshi_audio)} samples @ {SAMPLE_RATE} Hz)</td></tr>
    <tr><td style="padding: 2px 10px;">Model type</td><td>{MODEL_TYPE} | {n_params:.3f}B params ({n_nonzero:.3f}B non-zero)</td></tr>
    <tr><td style="padding: 2px 10px;">Seed</td><td>{SEED} ({SEED_TYPE})</td></tr>
    <tr><td style="padding: 2px 10px;">Speed</td><td>{ms_per_step:.0f} ms/step ({rtf:.2f}x real-time)</td></tr>
    <tr><td style="padding: 2px 10px;">Text tokens</td><td>{n_meaningful} meaningful / {len(text_tokens_arr)} total ({pad_ratio:.0%} silence)</td></tr>
  </table>
</div>
"""))

# Text transcript
if text_transcript:
    display(HTML(f"""
    <div style="background: #16213e; color: #a8d8ea; padding: 15px; border-radius: 8px; margin: 10px 0; font-family: monospace; font-size: 13px; max-height: 150px; overflow-y: auto;">
      <b>Inner Monologue (text stream):</b><br/>
      {text_transcript[:500]}{'...' if len(text_transcript) > 500 else ''}
    </div>
    """))

# ── Single audio player ──
display(HTML('<h3 style="color: #e94560;">Self-Play Audio</h3>'))
display(ipd.Audio(moshi_audio, rate=SAMPLE_RATE))

print(f"\nNote: LM model was moved to CPU. Re-run Cell 3 to regenerate.")

## Cell 6: Side-by-Side Comparison (Optional)

Load a previously saved audio sample from GDrive and play it alongside the
current variant for A/B listening.

**Usage:** Set `COMPARE_PATH` to the folder of a previous audio sample
(e.g., a teacher baseline generated with the same seed).

In [ ]:
import soundfile as sf

COMPARE_PATH = ""  # @param {type:"string"}
# Example: "/content/drive/MyDrive/moshilite/audio_samples/teacher_42_20260420_022500"

if COMPARE_PATH and os.path.isdir(COMPARE_PATH):
    compare_dir = Path(COMPARE_PATH)

    compare_meta_path = compare_dir / "metadata.json"
    if compare_meta_path.exists():
        with open(compare_meta_path) as f:
            compare_meta = json.load(f)
        compare_label = compare_meta.get("variant", "Unknown")
    else:
        compare_label = compare_dir.name
        compare_meta = {}

    compare_wav_path = compare_dir / "self_play.wav"
    if not compare_wav_path.exists():
        compare_wav_path = compare_dir / "moshi_channel.wav"
    if compare_wav_path.exists():
        compare_audio, compare_sr = sf.read(str(compare_wav_path), dtype="float32")

        display(HTML(f"""
        <div style="background: #1a1a2e; color: #eee; padding: 20px; border-radius: 12px; margin: 20px 0;">
          <h2 style="color: #e94560; margin-top: 0;">A/B Comparison</h2>
          <p>Same seed ({SEED}) — listen for quality differences</p>
        </div>
        """))

        display(HTML(f'<h3 style="color: #e94560;">A: {MODEL_VARIANT} (current)</h3>'))
        display(ipd.Audio(moshi_audio, rate=SAMPLE_RATE))

        display(HTML(f'<h3 style="color: #0f3460;">B: {compare_label} (saved)</h3>'))
        display(ipd.Audio(compare_audio, rate=compare_sr))

        if compare_meta:
            print(f"\nComparison details:")
            print(f"  A ({MODEL_VARIANT}):")
            print(f"    Params:  {n_params:.3f}B total, {n_nonzero:.3f}B non-zero")
            print(f"    Speed:   {ms_per_step:.0f} ms/step ({rtf:.2f}x RT)")
            print(f"  B ({compare_label}):")
            print(f"    Params:  {compare_meta.get('param_count_billions', '?')}B")
            print(f"    Speed:   {compare_meta.get('generation_speed_ms_per_step', '?')} ms/step "
                  f"({compare_meta.get('real_time_factor', '?')}x RT)")
    else:
        print(f"No audio file found in {COMPARE_PATH}")
else:
    print("No comparison path set. To compare:")
    print("   1. Generate & save a teacher baseline (Cell 7) with the same seed")
    print("   2. Set COMPARE_PATH to that folder and re-run this cell")

## Cell 7: Save Audio to Google Drive

Saves the self-play audio + metadata JSON to a timestamped folder on GDrive.

**Output:**
```
audio_samples/{variant}_{seed}_{timestamp}/
  ├── self_play.wav
  └── metadata.json
```

In [ ]:
import soundfile as sf
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
folder_name = f"{MODEL_VARIANT}_{SEED}_{timestamp}"
save_dir = Path(AUDIO_SAMPLES_DIR) / folder_name
save_dir.mkdir(parents=True, exist_ok=True)

# ── Save WAV file ──
print(f"Saving audio to: {save_dir}")
sf.write(str(save_dir / "self_play.wav"), moshi_audio, SAMPLE_RATE)

# ── Build metadata ──
metadata = {
    # Model identity
    "variant": MODEL_VARIANT,
    "model_type": MODEL_TYPE,
    "experiment_id": EXPERIMENT_ID,
    "model_path": MODEL_PATH,
    # Pruning details
    "pruning_type": variant_meta.get("pruning_type", "none"),
    "pruning_method": variant_meta.get("pruning_method", "none"),
    "sparsity": variant_meta.get("sparsity", 0.0),
    "kd_trained": variant_meta.get("kd_trained", False),
    "loss_config": variant_meta.get("loss_config", None),
    # Model size
    "param_count_billions": round(n_params, 3),
    "nonzero_param_billions": round(n_nonzero, 3),
    # Generation params
    "seed": SEED,
    "seed_type": SEED_TYPE,
    "duration_seconds": round(actual_duration, 1),
    "num_steps": NUM_STEPS,
    "temperature": TEMPERATURE,
    "temperature_text": TEMPERATURE_TEXT,
    "repetition_penalty": REPETITION_PENALTY,
    # Performance
    "generation_speed_ms_per_step": round(ms_per_step, 1),
    "real_time_factor": round(rtf, 2),
    "generation_wall_time_s": round(t_elapsed, 1),
    "gpu": torch.cuda.get_device_name(0),
    # Audio info
    "sample_rate": SAMPLE_RATE,
    "audio_samples": len(moshi_audio),
    # Text
    "text_transcript": text_transcript[:2000] if text_transcript else "",
    "text_tokens_total": len(text_tokens_arr),
    "text_tokens_meaningful": n_meaningful,
    "text_tokens_unique": n_unique,
    "pad_ratio": round(pad_ratio, 3),
    # Timestamp
    "timestamp": datetime.now().isoformat(),
}

with open(save_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

# ── Summary ──
total_size_mb = sum(f.stat().st_size for f in save_dir.iterdir()) / 1e6

print(f"\nSaved to Google Drive!")
print(f"   {save_dir}")
print(f"   self_play.wav   ({len(moshi_audio)/SAMPLE_RATE:.1f}s)")
print(f"   metadata.json")
print(f"   Total: {total_size_mb:.1f} MB")
print(f"\n   Tip: Use this path for A/B comparison:")
print(f"   COMPARE_PATH = \"{save_dir}\"")